# P4 — Projectile Motion

**Objective:** Simulate projectile motion with and without air resistance using numerical integration (Euler method). Produce two publication-quality plots comparing the two cases.

**Approach:** We treat the physics as a system of coupled ODEs and step forward in time using small increments (`time_step`). At each step we update position from velocity, then update velocity from acceleration (including drag if applicable).

**Two parts:**
- **Part A** — Trajectory plot (x vs y) for a fixed launch angle, both cases overlaid
- **Part B** — Range vs launch angle sweep (1°–89°), both cases overlaid, optimal angles labeled


## Imports
Standard scientific Python stack. `math` is used for scalar trig operations inside the loops.

In [ ]:
import math as m      # scalar math: cos, sin, sqrt, radians
import numpy as np     # array operations and argmax
import matplotlib.pyplot as plt  # plotting


## Parameters

All physical inputs are defined here so they can be changed in one place for a classroom demo.

**Drag model:** We model the projectile as a sphere. The drag force is:

$$F_{drag} = \frac{1}{2} \rho C_d A |v|^2$$

where $\rho$ is air density, $C_d$ is the drag coefficient, $A$ is cross-sectional area, and $|v|$ is total speed.

The cross-sectional area of a sphere is $A = \pi R^2$ (the circle you see head-on), **not** the surface area.

In [ ]:
# --- Launch conditions ---
v_0 = 25            # initial speed (m/s)
theta_launch = 45   # launch angle for Part A trajectory (degrees)

# --- Projectile physical properties ---
mass = 10           # mass (kg)
C_d = 0.47          # drag coefficient — 0.47 is standard for a sphere
R_sphere = 0.1      # radius of sphere (m)
rho_air = 1.225     # air density at sea level (kg/m^3)

# --- Initial position and time ---
x_0 = 0             # starting x position (m)
y_0 = 0             # starting y position (m)
start_time = 0      # starting time (s)
time_step = 0.01    # simulation time step (s) — smaller = more accurate

# --- Constant accelerations ---
a_x = 0             # no horizontal acceleration (ignoring wind)
a_y = -9.81         # gravitational acceleration (m/s^2, downward)

# --- Derived geometry ---
CS_Area = m.pi * R_sphere**2  # cross-sectional area of sphere (m^2)

# --- Decompose launch velocity into components ---
v_0x = v_0 * m.cos(m.radians(theta_launch))  # horizontal component (m/s)
v_0y = v_0 * m.sin(m.radians(theta_launch))  # vertical component (m/s)

print(f'Launch speed:       {v_0} m/s at {theta_launch}°')
print(f'Initial vx:         {v_0x:.2f} m/s')
print(f'Initial vy:         {v_0y:.2f} m/s')
print(f'Cross-sectional A:  {CS_Area:.5f} m^2')


## Equations of Motion

### Ideal case (no drag)
With no air resistance, horizontal velocity is constant and vertical velocity changes only due to gravity:

$$x(t) = v_{0x} t$$
$$y(t) = v_{0y} t - \frac{1}{2} g t^2$$

### With drag
Air resistance adds a force opposing motion, proportional to $v^2$. This gives us a system of ODEs we can't solve analytically, so we integrate numerically:

$$\frac{dv_x}{dt} = -\frac{\rho C_d A}{2m} v_x |v|$$

$$\frac{dv_y}{dt} = -g - \frac{\rho C_d A}{2m} v_y |v|$$

where $|v| = \sqrt{v_x^2 + v_y^2}$ is the total speed. Note that drag slows both components simultaneously — the $|v|$ term couples the x and y equations.

### Numerical integration (Euler method)
At each time step `dt`, we update position then velocity:

$$x_{n+1} = x_n + v_{x,n} \cdot dt$$
$$y_{n+1} = y_n + v_{y,n} \cdot dt$$
$$v_{x,n+1} = v_{x,n} + a_{x,n} \cdot dt$$
$$v_{y,n+1} = v_{y,n} + a_{y,n} \cdot dt$$

Position is updated **before** velocity so we use the velocity from the current step, not the next one.

## Part A — Trajectory

We run two separate simulations at the same launch angle (`theta_launch = 45°`): one with drag and one without. Both run until the projectile returns to `y = 0` (ground level). Results are stored as lists and converted to numpy arrays after the loop.

In [ ]:
# ── WITH DRAG ──────────────────────────────────────────────────────────────
A_DRAG_MATRIX = []          # will hold [time, x, y] for each step
time = start_time
curr_v_x = v_0x             # set initial velocity components
curr_v_y = v_0y
x_pos = x_0                 # set initial position
y_pos = y_0

A_DRAG_MATRIX.append([time, x_pos, y_pos])  # store launch point

while y_pos >= 0:           # run until projectile hits the ground
    x_pos = x_pos + curr_v_x * time_step   # update x position
    y_pos = y_pos + curr_v_y * time_step   # update y position

    v_mag = m.sqrt(curr_v_x**2 + curr_v_y**2)  # total speed — couples x and y drag

    # update velocity: gravity + drag deceleration in each component
    curr_v_x = curr_v_x + a_x * time_step - (0.5 * C_d * rho_air * CS_Area * curr_v_x * v_mag / mass * time_step)
    curr_v_y = curr_v_y + a_y * time_step - (0.5 * C_d * rho_air * CS_Area * curr_v_y * v_mag / mass * time_step)

    time += time_step
    A_DRAG_MATRIX.append([time, x_pos, y_pos])

A_DRAG_MATRIX = np.array(A_DRAG_MATRIX)   # convert list to numpy array for easy slicing

# ── WITHOUT DRAG ───────────────────────────────────────────────────────────
A_IDEAL_MATRIX = []         # will hold [time, x, y] for each step
time = start_time
curr_v_x = v_0x
curr_v_y = v_0y
x_pos = x_0
y_pos = y_0

A_IDEAL_MATRIX.append([time, x_pos, y_pos])  # store launch point

while y_pos >= 0:           # run until projectile hits the ground
    x_pos = x_pos + curr_v_x * time_step   # update x position
    y_pos = y_pos + curr_v_y * time_step   # update y position

    curr_v_x = curr_v_x + a_x * time_step  # no drag — only gravity affects velocity
    curr_v_y = curr_v_y + a_y * time_step

    time += time_step
    A_IDEAL_MATRIX.append([time, x_pos, y_pos])

A_IDEAL_MATRIX = np.array(A_IDEAL_MATRIX)


### Part A Plot
Both trajectories are plotted on the same axes. Landing points are marked and the range difference is annotated.

In [ ]:
drag_range = A_DRAG_MATRIX[-1, 1]       # x position at last recorded step = landing point
ideal_range = A_IDEAL_MATRIX[-1, 1]
range_diff = ideal_range - drag_range   # how much further the ideal case travels

plt.plot(A_IDEAL_MATRIX[:, 1], A_IDEAL_MATRIX[:, 2], label='No Drag')   # column 1 = x, column 2 = y
plt.plot(A_DRAG_MATRIX[:, 1], A_DRAG_MATRIX[:, 2], label='With Drag')
plt.scatter([ideal_range], [0], color='blue', zorder=5)     # mark ideal landing point
plt.scatter([drag_range], [0], color='orange', zorder=5)    # mark drag landing point
plt.annotate(f'Range difference: {range_diff:.1f} m', xy=(drag_range, 1), fontsize=9)
plt.title('Projectile Motion — Trajectory')
plt.xlabel('X Position (m)')
plt.ylabel('Y Position (m)')
plt.legend()
plt.grid(True)
plt.show()


## Part B — Range vs Launch Angle

We sweep launch angle from 1° to 89° in 1° steps. For each angle, we run both simulations (drag and no-drag) and record the horizontal range. This lets us find the optimal launch angle for each case.

**Expected result:** Without drag, the maximum range occurs at exactly 45°. With drag, the optimal angle shifts below 45° because a flatter trajectory spends less time fighting air resistance.

In [ ]:
B_DRAG_MATRIX = []    # stores [angle, range] for each angle with drag
B_IDEAL_MATRIX = []   # stores [angle, range] for each angle without drag

alpha = 1             # start at 1° (0° gives zero range)

while alpha < 90:     # sweep to 89° (90° launches straight up)

    # ── DRAG case for this angle ───────────────────────────────────────────
    x_pos, y_pos = x_0, y_0                        # reset position to origin
    curr_v_x = v_0 * m.cos(m.radians(alpha))       # horizontal velocity component
    curr_v_y = v_0 * m.sin(m.radians(alpha))       # vertical velocity component

    while y_pos >= 0:                               # integrate until landing
        x_pos = x_pos + curr_v_x * time_step
        y_pos = y_pos + curr_v_y * time_step
        v_mag = m.sqrt(curr_v_x**2 + curr_v_y**2)  # total speed for drag term
        curr_v_x = curr_v_x + a_x * time_step - (0.5 * C_d * rho_air * CS_Area * curr_v_x * v_mag / mass * time_step)
        curr_v_y = curr_v_y + a_y * time_step - (0.5 * C_d * rho_air * CS_Area * curr_v_y * v_mag / mass * time_step)

    B_DRAG_MATRIX.append([alpha, x_pos])            # store range for this angle

    # ── IDEAL case for this angle ──────────────────────────────────────────
    x_pos, y_pos = x_0, y_0
    curr_v_x = v_0 * m.cos(m.radians(alpha))
    curr_v_y = v_0 * m.sin(m.radians(alpha))

    while y_pos >= 0:                               # integrate until landing
        x_pos = x_pos + curr_v_x * time_step
        y_pos = y_pos + curr_v_y * time_step
        curr_v_x = curr_v_x + a_x * time_step      # no drag term
        curr_v_y = curr_v_y + a_y * time_step

    B_IDEAL_MATRIX.append([alpha, x_pos])

    alpha += 1    # increment angle by 1 degree

B_DRAG_MATRIX = np.array(B_DRAG_MATRIX)    # convert to numpy arrays for argmax
B_IDEAL_MATRIX = np.array(B_IDEAL_MATRIX)


### Part B Plot
Both range curves are overlaid. Vertical dashed lines mark the optimal angle for each case.

In [ ]:
ideal_opt_idx = np.argmax(B_IDEAL_MATRIX[:, 1])    # index of maximum range (no drag)
drag_opt_idx = np.argmax(B_DRAG_MATRIX[:, 1])      # index of maximum range (with drag)
ideal_opt_angle = B_IDEAL_MATRIX[ideal_opt_idx, 0] # corresponding angle (no drag)
drag_opt_angle = B_DRAG_MATRIX[drag_opt_idx, 0]    # corresponding angle (with drag)

plt.plot(B_IDEAL_MATRIX[:, 0], B_IDEAL_MATRIX[:, 1], label='No Drag')
plt.plot(B_DRAG_MATRIX[:, 0], B_DRAG_MATRIX[:, 1], label='With Drag')

# vertical dashed lines to highlight optimal angles
plt.axvline(x=ideal_opt_angle, color='blue', linestyle='--', alpha=0.5, label=f'Ideal optimum: {ideal_opt_angle:.0f}°')
plt.axvline(x=drag_opt_angle, color='orange', linestyle='--', alpha=0.5, label=f'Drag optimum: {drag_opt_angle:.0f}°')

plt.title('Range vs Launch Angle')
plt.xlabel('Launch Angle (degrees)')
plt.ylabel('Horizontal Distance (m)')
plt.legend()
plt.grid(True)
plt.show()
